In [1]:
import json
from pathlib import Path
import re

# huggingface dataset class
from datasets import Dataset

# rapidfire ai wrapper
from rapidfireai import Experiment
from rapidfireai.automl import List, RFGridSearch, RFModelConfig, RFLoraConfig, RFSFTConfig

In [2]:
def load_schema(db_id, schemas_dir='./schemas'):
    # change to the correct file name
    fname = db_id.replace(' ', '_').replace('/', '_') + '.json'
    # load the json file
    with open(f'{schemas_dir}/{fname}') as f:
        s = json.load(f)
    # to store table name and column mapping
    schema = {t: [] for t in s['table_names_original']}
    for tidx, cname in s['column_names_original']:
        # skip the synthetic '*' entry
        if tidx == -1:
            continue
        schema[s['table_names_original'][tidx]].append(cname)
    # {table_name: [col, col, ...]}
    return schema

In [3]:
def build_prompt(question, schema_text):
    return (
        f'Question:\n{question}\n'
        f'Schema:\n{schema_text}\n'
        'Return only schema_links JSON.'
    )

In [4]:
def formatting_function(row):
    SYSTEM_PROMPT = '''You are a schema-linking model.\n
    Given a database schema and a natural-language question, output only valid JSON.
    The JSON must map table names to lists of referenced column names.
    Include tables even when no specific columns are referenced, using an empty list.
    Do not include explanations, markdown, or extra text.

    Example output:
    {"TableName1": ["Col1", "Col2"], "TableName2": ["Col3"]}
    '''
    return {
        'prompt': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': row['prompt_text']}
        ],
        'completion': [
            {'role': 'assistant', 'content': row['completion_text']}
        ]
    }

In [5]:
def build_dataset(dataset_file, schemas_dir='./schemas'):
    # load the examples
    with open(dataset_file, 'r') as f:
        examples = json.load(f)
    # to store a list of formatted example for the dataset
    formatted = []
    for example in examples:
        # load the schema that corresponds to the db_id of the example
        schema = load_schema(db_id=example['db_id'], schemas_dir=schemas_dir)
        schema_text = '\n'.join(
            f'- {table}: {", ".join(columns)}'
            for table, columns in schema.items()
        )
        # build the prompt text by combining schema and question
        prompt_text = build_prompt(question=example['question'], schema_text=schema_text)
        # convert the target schema links into a JSON string
        completion_text = json.dumps(example['schema_links'], sort_keys=True)
        # add them to the dict
        example['prompt_text'] = prompt_text
        example['completion_text'] = completion_text
        # add the formatted example to the list
        formatted.append(example)
    return formatted

In [6]:
import re, json as _json

def _parse_json_output(text: str) -> dict:
    """Robustly parse model output: strip markdown fences, extract first {...} block."""
    text = text.strip()
    # Strip ```json ... ``` fences
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$",         "", text)
    # Extract first JSON object
    m = re.search(r"\{.*\}", text, re.DOTALL)
    if m:
        try:
            return _json.loads(m.group())
        except _json.JSONDecodeError:
            pass
    return {}


# from the project 2 statement
def _prf1(pred_set: set, gold_set: set):
    """Compute precision, recall, F1 for two sets."""
    if not gold_set and not pred_set:
        return 1.0, 1.0, 1.0
    if not gold_set:                      # empty-gold edge case
        return 0.0, 1.0, 0.0
    if not pred_set:
        return 0.0, 0.0, 0.0
    tp  = len(pred_set & gold_set)
    p   = tp / len(pred_set)
    r   = tp / len(gold_set)
    f1  = 2 * p * r / (p + r) if (p + r) else 0.0
    return p, r, f1


def schema_link_metrics(predictions: list[str], references: list[str]) -> dict:
    """
    Compute macro-averaged table and column P/R/F1.
    Both predictions and references are JSON strings mapping table->list[col].
    Mirrors the logic in the release packet eval.py.
    """
    t_p_list, t_r_list, t_f1_list = [], [], []
    c_p_list, c_r_list, c_f1_list = [], [], []

    for pred_str, gold_str in zip(predictions, references):
        pred_links = _parse_json_output(pred_str)
        gold_links = _json.loads(gold_str) if isinstance(gold_str, str) else gold_str

        # ---- Table-level ----
        pred_tables = {t.lower() for t in pred_links}
        gold_tables = {t.lower() for t in gold_links}
        tp, tr, tf = _prf1(pred_tables, gold_tables)
        t_p_list.append(tp); t_r_list.append(tr); t_f1_list.append(tf)

        # ---- Column-level (table.column pairs) ----
        pred_cols = {
            f"{t.lower()}.{c.lower()}"
            for t, cols in pred_links.items()
            for c in (cols or [])
        }
        gold_cols = {
            f"{t.lower()}.{c.lower()}"
            for t, cols in gold_links.items()
            for c in (cols or [])
        }
        cp, cr, cf = _prf1(pred_cols, gold_cols)
        c_p_list.append(cp); c_r_list.append(cr); c_f1_list.append(cf)

    def _mean(lst): return round(sum(lst) / len(lst), 4) if lst else 0.0

    PT, RT, F1T = _mean(t_p_list), _mean(t_r_list), _mean(t_f1_list)
    PC, RC, F1C = _mean(c_p_list), _mean(c_r_list), _mean(c_f1_list)
    table_score  = (PT + RT + F1T) / 3
    col_score    = (PC + RC + F1C) / 3
    leaderboard  = round(0.5 * table_score + 0.5 * col_score, 4)

    return {
        "table_P": PT, "table_R": RT, "table_F1": F1T,
        "col_P":   PC, "col_R":   RC, "col_F1":   F1C,
        "leaderboard_score": leaderboard,
    }


def compute_metrics_for_rf(eval_preds):
    """Adapter matching the signature expected by RFSFTConfig / TRL's compute_metrics."""
    predictions, labels = eval_preds
    # predictions and labels arrive as strings (decoded by RF trainer)
    return schema_link_metrics(list(predictions), list(labels))


print("Metrics function ready. Quick test:")
test_gold = '{"INJURY": ["AIS", "REGION"]}'
test_pred = test_gold   # perfect match
print(schema_link_metrics([test_pred], [test_gold]))  # expect 1
test_pred = '{"INJURY": ["AIS"]}'  # partial match
print(schema_link_metrics([test_pred], [test_gold])) # expect less than 1

Metrics function ready. Quick test:
{'table_P': 1.0, 'table_R': 1.0, 'table_F1': 1.0, 'col_P': 1.0, 'col_R': 1.0, 'col_F1': 1.0, 'leaderboard_score': 1.0}
{'table_P': 1.0, 'table_R': 1.0, 'table_F1': 1.0, 'col_P': 1.0, 'col_R': 0.5, 'col_F1': 0.6667, 'leaderboard_score': 0.8611}


In [ ]:
# LoRA PEFT configs lite with different adapter capacities
peft_configs_lite = List([
    # RFLoraConfig(
    #     # lora rank
    #     r=64,
    #     # normally 2 x rank
    #     lora_alpha=128,
    #     lora_dropout=0.05,
    #     # target on query, key, value projections and output projection
    #     target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    #     bias='none'
    # ),
    RFLoraConfig(
        # lora rank
        r=64,
        # normally 2 x rank
        lora_alpha=128,
        lora_dropout=0.05,
        # target on query, key, value projections and output projection
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 
                        'up_proj', 'down_proj', 'wte'],
        bias='none'
    )
])

config_set_lite = List([
    RFModelConfig(
        model_name='HuggingFaceTB/SmolLM2-1.7B-Instruct',
        peft_config=peft_configs_lite,
        training_args=RFSFTConfig(
            learning_rate=1e-5,
            lr_scheduler_type='cosine',
            per_device_train_batch_size=1,
            per_device_eval_batch_size=4,
            num_train_epochs=4,
            warmup_ratio=0.05,
            max_grad_norm=.2,
            gradient_accumulation_steps=4,
            logging_steps=2,
            eval_strategy='steps',
            eval_steps=4,
            fp16=True,
            save_strategy="steps",
            save_steps=16,
            save_total_limit=3,
            output_dir='./checkpoint'
        ),
        model_type='causal_lm',
        model_kwargs={'device_map': 'auto', 'torch_dtype': 'auto', 'use_cache': False},
        formatting_func=formatting_function,
        compute_metrics=compute_metrics_for_rf,
        generation_config={
            'max_new_tokens': 256,
            'temperature': 0.8,
            'top_p': 0.9,
            'top_k': 30,
            'repetition_penalty': 1.05,
        }
    ),
    RFModelConfig(
        model_name='Qwen/Qwen2.5-1.5B-Instruct',
        peft_config=peft_configs_lite,
        training_args=RFSFTConfig(
            learning_rate=1e-5,
            lr_scheduler_type='cosine',
            per_device_train_batch_size=1,
            per_device_eval_batch_size=4,
            num_train_epochs=4,
            warmup_ratio=0.05,
            max_grad_norm=.2,
            gradient_accumulation_steps=4,
            logging_steps=2,
            eval_strategy='steps',
            eval_steps=4,
            fp16=True,
            save_strategy="steps",
            save_steps=16,
            save_total_limit=3,
            output_dir='./checkpoint'
        ),
        model_type='causal_lm',
        model_kwargs={'device_map': 'auto', 'torch_dtype': 'auto', 'use_cache': False},
        formatting_func=formatting_function,
        compute_metrics=compute_metrics_for_rf,
        generation_config={
            'max_new_tokens': 256,
            'temperature': 0.8,
            'top_p': 0.9,
            'top_k': 30,
            'repetition_penalty': 1.05,
        }
    ),
    # RFModelConfig(
    #     model_name='TinyLlama/TinyLlama-1.1B-Chat-v1.0',
    #     peft_config=peft_configs_lite,
    #     training_args=RFSFTConfig(
    #         learning_rate=4e-5,
    #         lr_scheduler_type='cosine',
    #         per_device_train_batch_size=1,
    #         per_device_eval_batch_size=4,
    #         num_train_epochs=3,
    #         warmup_ratio=0.05,
    #         max_grad_norm=.2,
    #         gradient_accumulation_steps=4,
    #         logging_steps=2,
    #         eval_strategy='steps',
    #         eval_steps=4,
    #         fp16=True,
    #     ),
    #     model_type='causal_lm',
    #     model_kwargs={'device_map': 'auto', 'torch_dtype': 'auto', 'use_cache': False},
    #     formatting_func=formatting_function,
    #     compute_metrics=compute_metrics_for_rf,
    #     generation_config={
    #         'max_new_tokens': 256,
    #         'temperature': 0.8,
    #         'top_p': 0.9,
    #         'top_k': 30,
    #         'repetition_penalty': 1.05,
    #     }
    # )
])

RuntimeError: CUDA driver initialization failed, you might not have a CUDA gpu.

In [9]:
print(torch.cuda.get_device_name(0))

RuntimeError: CUDA driver initialization failed, you might not have a CUDA gpu.

In [ ]:
def create_model(model_config): 
    from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForMaskedLM
    # extract model configurations to initialize and load a pretrained model
    model_name = model_config['model_name']
    model_type = model_config['model_type']
    model_kwargs = model_config['model_kwargs']
    # load the pretrained model
    if model_type == 'causal_lm':
        model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
    elif model_type == 'seq2seq_lm':
        model = AutoModelForSeq2SeqLM.from_pretrained(model_name, **model_kwargs)
    elif model_type == 'masked_lm':
        model = AutoModelForMaskedLM.from_pretrained(model_name, **model_kwargs)
    else:
        return None
    # load the pretrained tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    return (model,tokenizer)

In [ ]:
config_group = RFGridSearch(
    configs=config_set_lite,
    trainer_type='SFT'
)

In [ ]:
# build training and valiation datasets
train_examples = build_dataset(dataset_file='../train.json', schemas_dir='../schemas')
val_examples = build_dataset(dataset_file='../validation.json', schemas_dir='../schemas')
# convert lists into HuggingFace Dataset objects
train_dataset = Dataset.from_list(train_examples)
val_dataset = Dataset.from_list(val_examples)

In [ ]:
print(formatting_function(train_dataset[0])['prompt'])
print(formatting_function(train_dataset[0])['completion'])

[{'role': 'system', 'content': 'You are a schema-linking model.\n\n    Given a database schema and a natural-language question, output only valid JSON.\n    The JSON must map table names to lists of referenced column names.\n    Include tables even when no specific columns are referenced, using an empty list.\n    Do not include explanations, markdown, or extra text.\n\n    Example output:\n    {"TableName1": ["Col1", "Col2"], "TableName2": ["Col3"]}\n    '}, {'role': 'user', 'content': 'Question:\nShow a count of injuries by body region where the injury severity is critical. The lookup code for critical injury is 5.\nSchema:\n- AIRBAG: CASEID, PSU, CASENO, CASENUMBER, CATEGORY, VEHNO, BAGNO, BAGLOCATION, BAGSTATUS, BAGTYPE, BAGDEPLOY, BAGMALFUNCTION, BAGFLAPSOPEN, BAGFLAPSDAM, BAGDAMAGE, BAGDAMSOURCE, PREVCRASH, PRIORMAINT, DEPLOYEVENT, BAGDVTOTAL, BAGDVLONG, BAGDVLAT, BAGDVENERGY, BAGDVSPEED, BAGDVBES, BAGDVEST, BAGDVRANK, CDCFORDEPLOY, CASEWGT, PSUSTRAT, VERSION\n- CHILDSEAT: CASEID

In [ ]:
# initialize a training experiment
experiment = Experiment(experiment_name='schema-linking-qwen-baseline', mode='fit')

The previously running experiment schema-linking-qwen-baseline_14 was forcibly ended. Created a new experiment 'schema-linking-qwen-baseline_15' with Experiment ID: 16 and Metric Experiment ID: 15 at /home/chc016/rapidfireai/rapidfire_experiments/schema-linking-qwen-baseline_15


In [ ]:
# run multi config training
experiment.run_fit(config_group, create_model_fn=create_model, train_dataset=train_dataset, eval_dataset=val_dataset, num_chunks=4, seed=42)

INFO 06-01 23:51:46 [__init__.py:216] Automatically detected platform cuda.
Started 1 worker processes successfully
Created workers
INFO 06-01 23:52:02 [__init__.py:216] Automatically detected platform cuda.


In [ ]:
# end the experiment
experiment.end()

Experiment schema-linking-qwen-baseline_12 ended
Workers stopped
